# 05: モデルの動作を収集・検証する

モデルの動作を記録し、採用するエピソードをデータセットに変換します。依存ライブラリ、学習済みモデル、LIBEROのシーンは実行時に取得します。

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `05_collection` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。動画が出たら、データに含める試行の番号を入力します。

同じフォルダに置くファイル

- `collection.json`
- `common.py`
- `libero_env.py`
- `prepare.py`
- `requirements.txt`
- `robot_io.py`
- `smolvla.py`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/05_collection"  # @param {type:"string"}
REQUIRED_FILES = [
    "collection.json",
    "common.py",
    "libero_env.py",
    "prepare.py",
    "requirements.txt",
    "robot_io.py",
    "smolvla.py",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_05")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## モデルを読み込み、行動を予測する処理

ここでは後のセルで使う処理を用意します。モデルを読み込むのは、実習を実行するセルです。

In [ ]:
MODEL_CODE = r"""
import torch
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla import processor_smolvla  # noqa: F401

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

from smolvla import checkpoint_inputs, observation_batch


def load_model(settings, image_size=256):
    checkpoint, config, vlm_repo = checkpoint_inputs(settings, image_size)
    policy = SmolVLAPolicy.from_pretrained(checkpoint, config=config, strict=True).eval()
    # 正規化・トークン化・行動の逆正規化はチェックポイント付属のものを使います。
    pre, post = make_pre_post_processors(
        config,
        pretrained_path=checkpoint,
        preprocessor_overrides={
            "device_processor": {"device": config.device},
            "tokenizer_processor": {"tokenizer_name": config.vlm_model_name},
        },
        postprocessor_overrides={"device_processor": {"device": "cpu"}},
    )
    return policy, pre, post, vlm_repo


def predict_action(policy, pre, post, obs, instruction, settings):
    batch = observation_batch(obs, instruction, settings)
    with torch.inference_mode():
        action = post(policy.select_action(pre(batch)))
    return action.squeeze(0).detach().cpu().numpy()

"""

## 収集するタスクと初期状態を確認する

In [ ]:
settings = json.loads((PROJECT / "collection.json").read_text())
print(json.dumps(settings, indent=2, ensure_ascii=False))

## 推論して、観測と行動を1フレームずつ記録する

In [ ]:
result = run_python(
    MODEL_CODE
    + r"""
from pathlib import Path

import imageio.v2 as imageio
import numpy as np

from common import output_directory, read_json, write_json
from libero_env import open_scene, scene_cases
from robot_io import state_vector, validate_action

settings = read_json("collection.json")
policy, pre, post, _ = load_model(settings["policy"])
folder = output_directory("collection")
episodes = []

for number, (task, initial) in enumerate(scene_cases(settings, "collection")):
    seed = settings["seed"] + task * 10000 + initial
    torch.manual_seed(seed)
    np.random.seed(seed)
    policy.reset()
    pre.reset()
    post.reset()
    env, obs, instruction = open_scene(settings, task, initial, seed)
    path = folder / f"episode_{number:03d}"
    path.mkdir()
    states, actions, front, wrist = [], [], [], []
    try:
        with imageio.get_writer(path / "video.mp4", fps=20) as video:
            for step in range(settings["max_steps"]):
                action = predict_action(policy, pre, post, obs, instruction, settings["policy"])
                # 収集方針を試す場合は、この場所でactionを調整します。
                action = validate_action(action)
                states.append(state_vector(obs))
                actions.append(action)
                front.append(obs["agentview_image"][::-1, ::-1].copy())
                wrist.append(obs["robot0_eye_in_hand_image"][::-1, ::-1].copy())
                video.append_data(front[-1])
                obs, _, done, _ = env.step(action)
                success = bool(env.check_success())
                if done or success:
                    break
            video.append_data(obs["agentview_image"][::-1, ::-1])
        np.savez_compressed(
            path / "frames.npz", state=states, action=actions, front=front, wrist=wrist
        )
        episodes.append({
            "episode": number, "task_id": task, "initial_state": initial,
            "instruction": instruction, "steps": len(actions), "success": success,
        })
    finally:
        env.close()
write_json(folder / "collection.json", {"episodes": episodes})
print(f"Results: {folder}", flush=True)
"""
)

## 動画で残したい試行を選ぶ

`episode_000` の番号は `0` です。指示どおりの動作ができたか確認し、残したい番号を次のセルに入力します。06で使う場合は3本以上選びます。

In [ ]:
from IPython.display import Video, display

collection_report = json.loads((result / "collection.json").read_text())
print(json.dumps(collection_report, indent=2, ensure_ascii=False))
for video in sorted(result.glob("episode_*/video.mp4")):
    print(video.parent.name)
    display(Video(str(video), embed=True))

In [ ]:
# 例: [0, 2, 3]。動画で確認した番号を入れて、このセルを実行します。
ACCEPT_EPISODES = []

## 採用した記録をLeRobot形式へ変換する

In [ ]:
if not ACCEPT_EPISODES or len(set(ACCEPT_EPISODES)) != len(ACCEPT_EPISODES):
    raise ValueError("採用するエピソード番号を重複なしで指定してください")
export_settings = {"folder": str(result), "episodes": ACCEPT_EPISODES}
(PROJECT / "export.json").write_text(json.dumps(export_settings))
dataset_result = run_python(r"""
from pathlib import Path

import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from common import read_json, write_json

settings = read_json("export.json")
folder = Path(settings["folder"])
available = {row["episode"]: row for row in read_json(folder / "collection.json")["episodes"]}
accepted = settings["episodes"]
if not set(accepted) <= available.keys():
    raise ValueError("存在しないエピソード番号です")

image = {"dtype": "image", "shape": (256, 256, 3), "names": ["height", "width", "channels"]}
dataset = LeRobotDataset.create(
    repo_id="workshop/collected_libero",
    root=folder / "dataset",
    fps=20,
    features={
        "observation.state": {"dtype": "float32", "shape": (8,), "names": None},
        "action": {"dtype": "float32", "shape": (7,), "names": None},
        "observation.images.image": image,
        "observation.images.image2": image,
    },
    robot_type="libero_panda",
    use_videos=False,
    video_backend="pyav",
)
try:
    for episode in accepted:
        with np.load(folder / f"episode_{episode:03d}" / "frames.npz") as frames:
            state, action = frames["state"], frames["action"]
            front, wrist = frames["front"], frames["wrist"]
            for index in range(len(action)):
                dataset.add_frame({
                    "observation.state": state[index],
                    "action": action[index],
                    "observation.images.image": front[index],
                    "observation.images.image2": wrist[index],
                    "task": available[episode]["instruction"],
                })
        dataset.save_episode()
finally:
    dataset.finalize()
write_json(folder / "dataset/workshop_provenance.json", {
    "episodes": [available[index] for index in accepted],
})
print(f"Results: {folder / 'dataset'}", flush=True)
""")
print("学習用データ:", dataset_result)

## 確認課題

採用したエピソードの番号と、動画を見て採用した理由を記録してください。

## 結果をDriveに保存する

収集結果と変換済みデータを保存します。変換後の試行番号は、入力した順に0から振り直します。表示されたデータのパスを06で使います。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = "05_collection_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)
for info in sorted(saved.glob("*/dataset/meta/info.json")):
    print("06のDATASET_ROOT:", info.parents[1])